In [6]:
import importlib, utils
importlib.reload(utils)
utils.build_merged.clear()
print("[hot reload] utils reloaded and build_merged cache cleared")


2026-03-24 16:25:00.679 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.679 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.680 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.681 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.682 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.682 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.683 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:25:00.683 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-

[hot reload] utils reloaded and build_merged cache cleared


# Inspect DB merge flow for overview table

The following python cell prints the exact code for build_merged and _compute_delegate_summary, then runs checks for delegate 13613.


In [7]:
import inspect
import pandas as pd
from utils import build_merged, _compute_delegate_summary, load_data, source_mtimes, load_remappings

print("=== build_merged code ===")
print(inspect.getsource(build_merged))
print("=== _compute_delegate_summary code ===")
print(inspect.getsource(_compute_delegate_summary))

df_p, df_i, df_abbrd = load_data(source_mtimes())
print("persons shape:", df_p.shape, "occurrences shape:", df_i.shape)
print("13613 in persons count:", (df_p["delegate_id"].astype(str) == "13613").sum())
print("duplicate delegate_ids in persons:", df_p["delegate_id"].astype(str).duplicated().sum())

remappings = load_remappings()
df_merged, n_placeholder, n_remapped, summary = build_merged(df_p, df_i, df_abbrd, extra_delegates=None, remappings=remappings, name_col="fullname")
print("build_merged result:", df_merged.shape, "placeholders", n_placeholder, "remapped", n_remapped)
print("summary shape:", summary.shape)
print("13613 in summary:", summary[summary["delegate_id"] == "13613"].shape)
print("duplicate delegate_id in summary:", summary["delegate_id"].duplicated().sum())

summary2 = _compute_delegate_summary(df_merged, df_p, "fullname")
print("recomputed summary shape:", summary2.shape)
print("13613 in recomputed summary:", summary2[summary2["delegate_id"] == "13613"].shape)
print("duplicate delegate_id recomputed:", summary2["delegate_id"].duplicated().sum())


2026-03-24 16:25:14.983 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


=== build_merged code ===
@st.cache_data(show_spinner="Merging data…", hash_funcs=_HASH_FUNCS, persist="disk")
def build_merged(
    df_p: pd.DataFrame,
    df_i: pd.DataFrame,
    df_bio: pd.DataFrame | None,
    extra_delegates: list[dict] | None = None,
    remappings: list[dict] | None = None,
    name_col: str = "fullname",
) -> tuple[pd.DataFrame, int, int, pd.DataFrame]:
    """Merge occurrences + persons, apply remappings, filter placeholders.

    Returns ``(merged_df, n_placeholder_rows, n_remapped_rows, summary_df)``.
    The summary (one row per delegate) is computed here so it shares this
    function's cache entry — no second hashing of the large merged DataFrame.
    """
    _n_placeholder_rows: int = 0
    _n_remapped_rows: int = 0

    persons = df_p.copy()
    if "delegate_id" in persons.columns:
        persons["delegate_id"] = persons["delegate_id"].astype(str)
    if "delegate_id" in df_i.columns:
        df_i = df_i.copy()
        df_i["delegate_id"] = df_i["deleg

In [8]:
assert "13613" in summary["delegate_id"].astype(str).values
assert summary["delegate_id"].astype(str).duplicated().sum() == 0
assert df_merged["delegate_id"].dtype == object
assert summary.equals(summary2) or summary[["delegate_id","n_occurrences"]].set_index("delegate_id").equals(summary2[["delegate_id","n_occurrences"]].set_index("delegate_id"))

In [9]:
df_merged

,Kolom1,offset,end,class,pattern,oud id,delegate_id,delegate_name,delegate_score,d,...,resolutie_refs,minjaar,maxjaar,pattern_p,heerlijkheid,birth_year,death_year,name_mismatch,pattern_is_valid,age_at_event
0,427232,353,360,delegate,Jordens,None,236,"Jordens, Gerrit Dqavid",0.0,17,...,NaN,1795.0,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,NaN,NaN,NaN,False,True,NaN
1,427245,276,283,delegate,Jordens,None,236,"Jordens, Gerrit Dqavid",0.0,18,...,NaN,1795.0,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,NaN,NaN,NaN,False,True,NaN
2,427257,225,232,delegate,Jordens,None,236,"Jordens, Gerrit Dqavid",0.0,19,...,NaN,1795.0,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,NaN,NaN,NaN,False,True,NaN
3,427271,261,268,delegate,Jordens,None,236,"Jordens, Gerrit Dqavid",0.0,20,...,NaN,1795.0,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,NaN,NaN,NaN,False,True,NaN
4,427282,278,285,delegate,Jordens,None,236,"Jordens, Gerrit Dqavid",0.0,23,...,NaN,1795.0,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,NaN,NaN,NaN,False,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420742,287913,237,240,delegate,Can,None,21971,van Canter,0.0,6,...,NaN,1760.0,1760.0,van Canter;Van Canter;Can;Chan,NaN,NaN,NaN,False,True,NaN
420743,359725,505,508,delegate,Can,None,21971,van Canter,0.0,19,...,NaN,1760.0,1760.0,van Canter;Van Canter;Can;Chan,NaN,NaN,NaN,False,True,NaN
420744,359799,387,390,delegate,Can,None,21971,van Canter,0.0,23,...,NaN,1760.0,1760.0,van Canter;Van Canter;Can;Chan,NaN,NaN,NaN,False,True,NaN
420745,394980,434,437,delegate,Can,None,21971,van Canter,0.0,13,...,NaN,1760.0,1760.0,van Canter;Van Canter;Can;Chan,NaN,NaN,NaN,False,True,NaN


In [10]:
summary

,fullname,delegate_id,n_occurrences,first_year,last_year,n_patterns,n_alive_flags,n_name_mismatches,max_gap_years
0,"Abbink, Bernard Engelbert",18744,4.0,1787.0,1787.0,2.0,0.0,4.0,0
1,"Ablaing, Johan Daniël d'Giessenburg en Haulchi...",18379,4576.0,1726.0,1775.0,252.0,0.0,0.0,21
2,"Acquet, Hendrik Georgesz. d'",13591,141.0,1755.0,1758.0,3.0,0.0,0.0,1
3,"Alberda van Bloemersma en Bijma, Edzard Reint",17565,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,"Alberda van Resuma, Mello",<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...
946,None,-1,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
947,None,-20,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
948,None,18451,5.0,1713.0,1721.0,5.0,0.0,5.0,4
949,"Piccardt, Jan",17544,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [11]:
df_merged.shape

(420747, 37)

In [12]:
import importlib, utils
importlib.reload(utils)
utils.build_merged.clear()

from utils import build_merged, _compute_delegate_summary, load_data, source_mtimes, load_remappings

df_p, df_i, df_abbrd = load_data(source_mtimes())
remappings = load_remappings()

df_merged, n_placeholder, n_remapped, summary = build_merged(
    df_p, df_i, df_abbrd,
    extra_delegates=None,
    remappings=remappings,
    name_col="fullname"
)

summary2 = _compute_delegate_summary(df_merged, df_p, "fullname")

print("build_merged rows:", df_merged.shape[0], "cols:", df_merged.shape[1])
print("n_placeholder:", n_placeholder, "n_remapped:", n_remapped)
print("summary rows:", summary.shape[0], "cols:", summary.shape[1])
print("summary2 rows:", summary2.shape[0], "cols:", summary2.shape[1])
print("13613 in summary:", summary[summary["delegate_id"] == "13613"].shape)
print("13613 in summary2:", summary2[summary2["delegate_id"] == "13613"].shape)
print("dup delegate_id summary:", summary["delegate_id"].duplicated().sum())
print("dup delegate_id summary2:", summary2["delegate_id"].duplicated().sum())
print("merged delegate_id dtype:", df_merged["delegate_id"].dtype)
print("persons delegate_id dtype:", df_p["delegate_id"].dtype)

print("\n=== merged sample for 13613 ===")
print(df_merged[df_merged["delegate_id"] == "13613"].head(5).to_string(index=False))

print("\n=== summary for 13613 ===")
print(summary[summary["delegate_id"] == "13613"].to_string(index=False))

2026-03-24 16:29:09.758 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.759 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.759 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.761 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.761 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.762 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.762 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-24 16:29:09.762 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-

build_merged rows: 420747 cols: 37
n_placeholder: 2161 n_remapped: 84
summary rows: 951 cols: 9
summary2 rows: 951 cols: 9
13613 in summary: (1, 9)
13613 in summary2: (1, 9)
dup delegate_id summary: 0
dup delegate_id summary2: 0
merged delegate_id dtype: object
persons delegate_id dtype: Int64

=== merged sample for 13613 ===
Empty DataFrame
Columns: [Kolom1, offset, end, class, pattern, oud id, delegate_id, delegate_name, delegate_score, d, m, j, lowerpattern, name, namens, status, opmerkingen, niettevinden, correcties, cons_id_str, fullname, voornaam, tussenvoegsel, geslachtsnaam, geboortejaar, overlijdensjaar, provincie, resolutie_refs, minjaar, maxjaar, pattern_p, heerlijkheid, birth_year, death_year, name_mismatch, pattern_is_valid, age_at_event]
Index: []

=== summary for 13613 ===
                         fullname delegate_id  n_occurrences  first_year  last_year  n_patterns  n_alive_flags  n_name_mismatches  max_gap_years
Berkhout, Pieter Jacob Teding van       13613           

In [13]:
summary

,fullname,delegate_id,n_occurrences,first_year,last_year,n_patterns,n_alive_flags,n_name_mismatches,max_gap_years
0,"Abbink, Bernard Engelbert",18744,4.0,1787.0,1787.0,2.0,0.0,4.0,0
1,"Ablaing, Johan Daniël d'Giessenburg en Haulchi...",18379,4576.0,1726.0,1775.0,252.0,0.0,0.0,21
2,"Acquet, Hendrik Georgesz. d'",13591,141.0,1755.0,1758.0,3.0,0.0,0.0,1
3,"Alberda van Bloemersma en Bijma, Edzard Reint",17565,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,"Alberda van Resuma, Mello",<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...
946,None,-1,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
947,None,-20,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
948,None,18451,5.0,1713.0,1721.0,5.0,0.0,5.0,4
949,"Piccardt, Jan",17544,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [14]:
summary[summary["delegate_id"] == "13613"]

,fullname,delegate_id,n_occurrences,first_year,last_year,n_patterns,n_alive_flags,n_name_mismatches,max_gap_years
950,"Berkhout, Pieter Jacob Teding van",13613,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [15]:
assert summary.loc[summary['delegate_id']=='13613', 'n_occurrences'].item() > 0
assert (summary['delegate_id'].astype(str).duplicated().sum() == 0)

AssertionError: 

In [16]:
assert summary.loc[summary['delegate_id']=='13613', 'n_occurrences'].item() > 0
assert (summary['delegate_id'].astype(str).duplicated().sum() == 0)

AssertionError: 